# Zolai Language Learner v1
**Deep Linguistic Analysis from Bible Parallel Dataset**

Learns Zolai grammar, tenses, word order, particles, and phrase patterns by comparing with English Bible text.

### What This Notebook Does:
1. **Grammar Extraction** — Identifies tense markers, particles, word order patterns
2. **Phrase Mapping** — Maps Zolai phrases to English equivalents
3. **Tense Analysis** — Learns how Zolai expresses past/present/future
4. **POS-like Tagging** — Categorizes words by function (verb, noun, particle, etc.)
5. **Sentence Structure** — Analyzes SVO/SOV patterns, clause structure
6. **Export Enhanced Model** — Creates improved language model for crawler and training

**Input:** `bible_zolai_english_pairs.jsonl` from Bible pipeline
**Output:** Enhanced vocabulary, grammar rules, phrase mappings

In [ ]:
#%% Cell 1: Environment Check & Dependencies
import os, sys, platform, json, hashlib, re
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import socket

print(f"Platform: {platform.platform()}")
print(f"Python:   {sys.version}")
print(f"CWD:      {os.getcwd()}")

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else ".").resolve()
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")

def has_internet():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=3)
        return True
    except OSError:
        return False

INTERNET = has_internet()
print(f"Internet: {'ON' if INTERNET else 'OFF'}")

if INTERNET:
    !pip install -q -U "pandas>=2.0,<3" "matplotlib>=3.5,<3.10.1" "tqdm>=4.66" "datasets>=2.19.0"
    print("Install complete.")

import pandas as pd
from tqdm.auto import tqdm

print("\nAll imports OK")

In [ ]:
#%% Cell 2: Load Bible Parallel Data

# Find Bible dataset
BIBLE_DATA = None
if IS_KAGGLE:
    for candidate in [
        Path("/kaggle/input/datasets/peterpausianlian/bible-datasets/zolai_bible_dataset"),
        Path("/kaggle/input/datasets/peterpausianlian/bible-datasets/zolai_bible_dataset/../Chin-Bible"),
    ]:
        if (candidate / "bible_zolai_english_pairs.jsonl").exists():
            BIBLE_DATA = candidate
            break
else:
    BIBLE_DATA = Path(os.getcwd()).resolve() / "zolai_bible_pipeline_out"

if BIBLE_DATA is None or not (BIBLE_DATA / "bible_zolai_english_pairs.jsonl").exists():
    raise FileNotFoundError(
        "Bible dataset not found. Run zolai-bible-pipeline-v1.ipynb first."
    )

print(f"Bible data: {BIBLE_DATA}")

# Working directory
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else ".").resolve()

# Output directory for language model
OUTPUT_DIR = WORK_DIR / "zolai_language_model"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# Load parallel pairs
pairs = []
with open(BIBLE_DATA / "bible_zolai_english_pairs.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Loaded {len(pairs):,} Zolai↔English pairs")

# Also load language patterns
patterns = None
if (BIBLE_DATA / "language_patterns.json").exists():
    with open(BIBLE_DATA / "language_patterns.json", 'r', encoding='utf-8') as f:
        patterns = json.load(f)
    print(f"Loaded language patterns")

# Also load full parallel corpus for multi-translation analysis
parallel_corpus = []
if (BIBLE_DATA / "bible_parallel.jsonl").exists():
    with open(BIBLE_DATA / "bible_parallel.jsonl", 'r', encoding='utf-8') as f:
        for line in f:
            parallel_corpus.append(json.loads(line))
    print(f"Loaded {len(parallel_corpus):,} parallel verses")

# Sample
print(f"\nSample pair:")
for p in pairs[:2]:
    print(f"  Zolai:   {p['source'][:80]}")
    print(f"  English: {p['target'][:80]}")
    print()

In [ ]:
#%% Cell 3: Tense Marker Extraction
# Verb-focused tense/aspect analysis based on Bible pairs and Zolai grammar notes.

print("=" * 60)
print("TENSE MARKER ANALYSIS")
print("=" * 60)

def normalize_token(token):
    token = unicodedata.normalize("NFKC", str(token).lower())
    token = token.replace("’", "'").replace("‘", "'").replace("`", "'")
    token = re.sub(r"^[^a-z']+|[^a-z']+$", "", token)
    token = token.replace("'", "")
    return token

tense_indicators = {
    "past": ["was", "were", "had", "did", "said", "made", "created", "called", "went", "came", "gave", "put", "built", "killed", "saw", "heard", "spoke"],
    "present": ["is", "are", "has", "have", "does", "says", "makes", "comes", "goes", "lives", "stands", "knows", "sees", "hears"],
    "future": ["will", "shall", "going to", "about to"],
    "imperative": ["let", "be", "go", "come", "hear", "see", "know", "keep", "make", "take"],
    "perfect": ["has been", "had been", "have been", "has done", "had done"],
}

# Curated from Bible usage plus Zolai Sinna / Gelhmaan style verb patterns.
VERB_LEXICON = {
    "om", "ahi", "hi", "ci", "gen", "hilh", "sim", "gelh", "sin", "thei", "muh", "mu",
    "pai", "hong", "lut", "suan", "tung", "zui", "nei", "pia", "piak", "lak", "ngah",
    "bawl", "sem", "sep", "koih", "koi", "khen", "khenkhia", "kikhawm", "piang", "piangsak",
    "gamtang", "vaksak", "sak", "damsak", "thuak", "thum", "bia", "thunget", "duh", "it",
    "hua", "thi", "dam", "nuntak", "zo", "ciah", "bei", "khia", "gawt", "huh", "phat",
    "zah", "za", "dong", "dawh", "pek", "pe", "khawm", "thupiak", "up", "kinep", "kizui",
}
VERB_SUFFIXES = {"hi", "ding", "sak", "khia", "lut", "pai", "ciah", "zo", "mu", "ci", "piak", "thei", "sim", "gelh", "bawl", "sem", "nei", "zui", "ngen", "thum"}
TENSE_PARTICLES = {"ding", "dingin", "ciangin", "khit", "khitciangin", "lai", "zong", "manin", "hileh", "leh", "lo", "hiam"}
IGNORE_WORDS = {
    "in", "uh", "a", "na", "ka", "kei", "amah", "ama", "amaute", "tua", "pen", "mah",
    "tawh", "panin", "hangin", "sungah", "kiangah", "topa", "pasian", "thu", "mite", "mi",
    "ahih", "hih", "hiam", "note", "bangin", "khempeuh", "hi", "ahi", "lo",
}


def candidate_verbs(zolai_text):
    tokens = [normalize_token(tok) for tok in re.findall(r"[A-Za-z'\-]+", zolai_text)]
    tokens = [tok for tok in tokens if tok]
    verbs = []
    for tok in tokens:
        if tok in IGNORE_WORDS:
            continue
        if tok in VERB_LEXICON or any(tok.endswith(suf) for suf in VERB_SUFFIXES):
            verbs.append(tok)
    return verbs, tokens


tense_cooccurrence = {tense: Counter() for tense in tense_indicators}
tense_particles = {tense: Counter() for tense in tense_indicators}
tense_sentence_context = {tense: [] for tense in tense_indicators}

for pair in pairs:
    english_lower = pair["target"].lower()
    verbs, tokens = candidate_verbs(pair["source"])
    particles_here = [tok for tok in tokens if tok in TENSE_PARTICLES]
    for tense, indicators in tense_indicators.items():
        if any(ind in english_lower for ind in indicators):
            for verb in verbs:
                tense_cooccurrence[tense][verb] += 1
            for particle in particles_here:
                tense_particles[tense][particle] += 1
            if len(tense_sentence_context[tense]) < 60:
                tense_sentence_context[tense].append({
                    "zolai": pair["source"],
                    "english": pair["target"],
                    "verse_id": pair.get("verse_id", ""),
                    "verbs": verbs[:12],
                    "particles": particles_here[:12],
                })

tense_markers = {}
for tense, counter in tense_cooccurrence.items():
    top_words = counter.most_common(25)
    tense_markers[tense] = top_words
    print(f"\n{tense.upper()} tense - likely Zolai verbs:")
    for word, count in top_words[:15]:
        print(f"  {word:20s} ({count:4d})")
    particle_view = tense_particles[tense].most_common(10)
    if particle_view:
        print("  particles:")
        for word, count in particle_view[:8]:
            print(f"    {word:18s} ({count:4d})")

likely_particles = sorted({word for counter in tense_particles.values() for word, count in counter.items() if count >= 10})
print(f"\nLikely tense/aspect particles: {', '.join(likely_particles)}")

tense_context_file = OUTPUT_DIR / "tense_context_samples.json"
with open(tense_context_file, "w", encoding="utf-8") as f:
    json.dump({tense: ctx[:20] for tense, ctx in tense_sentence_context.items()}, f, ensure_ascii=False, indent=2)
print(f"\nSaved tense context samples: {tense_context_file}")


In [ ]:
#%% Cell 4: Word Order & Sentence Structure Analysis

print("=" * 60)
print("WORD ORDER & SENTENCE STRUCTURE")
print("=" * 60)

# Analyze word order patterns
# English is SVO (Subject-Verb-Object)
# Let's see what Zolai does

def analyze_word_order(zolai_text, english_text):
    """
    Compare word order between Zolai and English.
    Returns alignment hints.
    """
    z_words = zolai_text.lower().split()
    e_words = english_text.lower().split()
    
    # Find common content words (skip articles, prepositions)
    skip_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'of', 'for', 'with'}
    
    z_content = [(i, w) for i, w in enumerate(z_words) if len(w) > 2]
    e_content = [(i, w) for i, w in enumerate(e_words) if w not in skip_words and len(w) > 2]
    
    return z_content, e_content

# Collect first/last word patterns
zolai_first_words = Counter()
zolai_last_words = Counter()
english_first_words = Counter()
english_last_words = Counter()

# Sentence length comparison
zolai_lengths = []
english_lengths = []

for p in pairs:
    z_words = p['source'].split()
    e_words = p['target'].split()
    
    if z_words:
        zolai_first_words[z_words[0].lower()] += 1
        zolai_last_words[z_words[-1].lower().rstrip('.')] += 1
    if e_words:
        english_first_words[e_words[0].lower()] += 1
        english_last_words[e_words[-1].lower().rstrip('.')] += 1
    
    zolai_lengths.append(len(z_words))
    english_lengths.append(len(e_words))

print(f"\nZolai sentence length: mean={sum(zolai_lengths)/len(zolai_lengths):.1f}, "
      f"min={min(zolai_lengths)}, max={max(zolai_lengths)}")
print(f"English sentence length: mean={sum(english_lengths)/len(english_lengths):.1f}, "
      f"min={min(english_lengths)}, max={max(english_lengths)}")

print(f"\nTop 15 Zolai sentence starters:")
for word, count in zolai_first_words.most_common(15):
    print(f"  {word:20s} ({count:4d})")

print(f"\nTop 15 Zolai sentence enders:")
for word, count in zolai_last_words.most_common(15):
    print(f"  {word:20s} ({count:4d})")

print(f"\nTop 15 English sentence starters:")
for word, count in english_first_words.most_common(15):
    print(f"  {word:20s} ({count:4d})")

In [ ]:
#%% Cell 5: Phrase Mapping — Zolai ↔ English

print("=" * 60)
print("PHRASE MAPPING")
print("=" * 60)

def extract_phrases(text, min_words=2, max_words=4):
    """Extract n-gram phrases from text."""
    words = text.lower().split()
    phrases = []
    for n in range(min_words, max_words + 1):
        for i in range(len(words) - n + 1):
            phrases.append(' '.join(words[i:i+n]))
    return phrases

# Build phrase co-occurrence map
phrase_pairs = defaultdict(list)
zolai_phrase_freq = Counter()
english_phrase_freq = Counter()
verse_phrase_links = defaultdict(lambda: defaultdict(set))

for p in pairs:
    z_phrases = extract_phrases(p['source'])
    e_phrases = extract_phrases(p['target'])
    
    for zp in z_phrases:
        zolai_phrase_freq[zp] += 1
    for ep in e_phrases:
        english_phrase_freq[ep] += 1
    
    # Link phrases by verse
    for zp in z_phrases:
        phrase_pairs[zp].extend(e_phrases)
        for ep in e_phrases:
            verse_phrase_links[zp][ep].add(p.get('verse_id', ''))

# Find most common Zolai phrases and their English equivalents
print("\nTop 50 Zolai phrases with English equivalents:")
phrase_mapping = {}
for zp, count in zolai_phrase_freq.most_common(50):
    if count < 3:
        continue
    
    # Find most common English equivalent
    english_equivalents = Counter(phrase_pairs[zp])
    top_english = english_equivalents.most_common(5)
    
    # Get verses where this phrase appears
    example_verses = []
    for p in pairs:
        if zp in p['source'].lower():
            example_verses.append({
                'zolai': p['source'],
                'english': p['target'],
                'verse_id': p.get('verse_id', ''),
            })
            if len(example_verses) >= 3:
                break
    
    phrase_mapping[zp] = {
        'count': count,
        'english': top_english,
        'examples': example_verses,
    }
    
    english_str = ', '.join(f"'{e}' ({c})" for e, c in top_english[:3])
    print(f"  '{zp}' ({count}x) -> {english_str}")

print(f"\nTotal phrase mappings: {len(phrase_mapping)}")

# Save phrase mapping with examples
phrase_file = OUTPUT_DIR / "phrase_mapping.json"
with open(phrase_file, 'w', encoding='utf-8') as f:
    json.dump(phrase_mapping, f, ensure_ascii=False, indent=2)
print(f"Saved phrase mapping: {phrase_file}")

In [ ]:
#%% Cell 6: POS-like Word Categorization

print("=" * 60)
print("WORD CATEGORIZATION (POS-like)")
print("=" * 60)

# Categorize Zolai words by their behavior
all_zolai_words = Counter()
all_english_words = Counter()
word_pairs = defaultdict(Counter)
word_contexts = defaultdict(list)

for p in pairs:
    z_words = p['source'].lower().split()
    e_words = p['target'].lower().split()
    
    for zw in z_words:
        all_zolai_words[zw] += 1
    for ew in e_words:
        all_english_words[ew] += 1
    
    # Simple word alignment by position
    min_len = min(len(z_words), len(e_words))
    for i in range(min_len):
        word_pairs[z_words[i]][e_words[i]] += 1

# Categorize by word properties
categories = {
    'particles': [],
    'content_words': [],
    'proper_nouns': [],
    'numbers': [],
}

for word, count in all_zolai_words.most_common():
    if len(word) <= 2 and count > 100:
        categories['particles'].append((word, count))
    elif word[0].isupper():
        categories['proper_nouns'].append((word, count))
    elif word.isdigit():
        categories['numbers'].append((word, count))
    elif len(word) > 2:
        categories['content_words'].append((word, count))

# Enhanced particle analysis
print(f"\nParticles ({len(categories['particles'])} words, high frequency):")
particle_details = []
for word, count in categories['particles'][:40]:
    eng_equiv = word_pairs[word].most_common(5)
    eng_str = ', '.join(f"{e}({c})" for e, c in eng_equiv)
    
    # Determine likely function
    likely_function = 'unknown'
    if word in ['leh', 'le', 'a']:
        likely_function = 'conjunction/linker'
    elif word in ['hi', 'hih', 'hiam']:
        likely_function = 'sentence particle'
    elif word in ['in', 'a', 'ka', 'na']:
        likely_function = 'marker/particle'
    elif word in ['ci', 'ciang', 'ci-in']:
        likely_function = 'quotative/complementizer'
    
    particle_details.append({
        'word': word,
        'count': count,
        'english_equiv': eng_equiv,
        'function': likely_function,
    })
    print(f"  {word:15s} ({count:5d}) [{likely_function:25s}] -> {eng_str}")

# Content word analysis
print(f"\nTop 50 Content Words:")
content_word_details = []
for word, count in categories['content_words'][:50]:
    eng_equiv = word_pairs[word].most_common(5)
    eng_str = ', '.join(f"{e}({c})" for e, c in eng_equiv)
    
    content_word_details.append({
        'word': word,
        'count': count,
        'english_equiv': eng_equiv,
    })
    print(f"  {word:20s} ({count:5d}) -> {eng_str}")

# Save POS data
pos_file = OUTPUT_DIR / "pos_categorization.json"
with open(pos_file, 'w', encoding='utf-8') as f:
    json.dump({
        'particles': particle_details,
        'content_words': content_word_details,
        'proper_nouns': dict(categories['proper_nouns'][:50]),
    }, f, ensure_ascii=False, indent=2)
print(f"\nSaved POS categorization: {pos_file}")

In [ ]:
#%% Cell 7: Grammar Rule Extraction

print("=" * 60)
print("GRAMMAR RULE EXTRACTION")
print("=" * 60)

# Extract grammar patterns
grammar_rules = {
    'sentence_enders': [],
    'common_prefixes': [],
    'common_suffixes': [],
    'conjunction_patterns': [],
    'negation_patterns': [],
    'question_patterns': [],
}

# 1. Sentence enders (particles that end sentences)
sentence_enders = Counter()
for p in pairs:
    words = p['source'].split()
    if words:
        last_word = words[-1].lower().rstrip('.!?')
        sentence_enders[last_word] += 1

grammar_rules['sentence_enders'] = sentence_enders.most_common(10)
print(f"\nSentence-ending particles:")
for word, count in grammar_rules['sentence_enders']:
    print(f"  {word:15s} ({count:5d})")

# 2. Common prefixes/suffixes
prefixes = Counter()
suffixes = Counter()
for word in all_zolai_words:
    if len(word) >= 4:
        prefixes[word[:2]] += 1
        prefixes[word[:3]] += 1
        suffixes[word[-2:]] += 1
        suffixes[word[-3:]] += 1

grammar_rules['common_prefixes'] = prefixes.most_common(15)
grammar_rules['common_suffixes'] = suffixes.most_common(15)

print(f"\nCommon prefixes:")
for pref, count in grammar_rules['common_prefixes']:
    print(f"  {pref:10s} ({count:5d})")

print(f"\nCommon suffixes:")
for suff, count in grammar_rules['common_suffixes']:
    print(f"  {suff:10s} ({count:5d})")

# 3. Conjunction patterns
conjunctions = {' leh ', ' le ', ' a ', ' ci ', ' hi ', ' in ', ' pen '}
conj_patterns = Counter()
for p in pairs:
    text = ' ' + p['source'].lower() + ' '
    for conj in conjunctions:
        if conj in text:
            conj_patterns[conj.strip()] += 1

grammar_rules['conjunction_patterns'] = conj_patterns.most_common()
print(f"\nConjunction/linker patterns:")
for conj, count in grammar_rules['conjunction_patterns']:
    print(f"  '{conj}' ({count:5d})")

# 4. Negation patterns
negation_words = Counter()
for p in pairs:
    english_lower = p['target'].lower()
    if 'not' in english_lower or 'no ' in english_lower or 'never' in english_lower:
        z_words = p['source'].lower().split()
        for zw in z_words:
            negation_words[zw] += 1

grammar_rules['negation_patterns'] = negation_words.most_common(10)
print(f"\nNegation patterns (Zolai words in negative sentences):")
for word, count in grammar_rules['negation_patterns']:
    print(f"  {word:15s} ({count:5d})")

# 5. Question patterns
question_words = Counter()
for p in pairs:
    if p['target'].rstrip().endswith('?'):
        z_words = p['source'].lower().split()
        for zw in z_words:
            question_words[zw] += 1

grammar_rules['question_patterns'] = question_words.most_common(10)
print(f"\nQuestion patterns (Zolai words in questions):")
for word, count in grammar_rules['question_patterns']:
    print(f"  {word:15s} ({count:5d})")

In [ ]:
#%% Cell 8: Multi-Translation Vocabulary Enrichment

print("=" * 60)
print("MULTI-TRANSLATION VOCABULARY ENRICHMENT")
print("=" * 60)

# Use multiple Zolai translations to find word variations
if parallel_corpus:
    # Find verses with multiple Zolai translations
    zolai_keys = ['tdb77', 'tedim1932']
    
    word_variations = defaultdict(set)  # base_word -> {variations}
    translation_pairs_counter = Counter()
    
    for verse in parallel_corpus:
        tdb = verse.get('tdb77', '')
        ted = verse.get('tedim1932', '')
        
        if tdb and ted:
            tdb_words = tdb.lower().split()
            ted_words = ted.lower().split()
            
            # Find words that differ between translations
            min_len = min(len(tdb_words), len(ted_words))
            for i in range(min_len):
                if tdb_words[i] != ted_words[i]:
                    # These are likely variant spellings
                    w1, w2 = sorted([tdb_words[i], ted_words[i]])
                    key = f"{w1}|{w2}"
                    translation_pairs_counter[key] += 1
    
    print(f"\nWord variations between TDB77 and Tedim 1932:")
    for pair, count in translation_pairs_counter.most_common(30):
        w1, w2 = pair.split('|')
        print(f"  {w1:20s} <-> {w2:20s} ({count:4d}x)")
    
    # Build normalization map
    normalization_map = {}
    for pair, count in translation_pairs_counter.most_common():
        if count >= 3:  # Only frequent variations
            w1, w2 = pair.split('|')
            # Use the more common form as canonical
            normalization_map[w2] = w1
    
    print(f"\nNormalization map ({len(normalization_map)} entries):")
    for variant, canonical in list(normalization_map.items())[:20]:
        print(f"  {variant:20s} -> {canonical}")

In [ ]:
#%% Cell 9: Export Enhanced Language Model

OUTPUT_DIR = WORK_DIR / "zolai_language_model"
OUTPUT_DIR.mkdir(exist_ok=True)

# Build comprehensive language model
language_model = {
    'metadata': {
        'source': 'zolai-language-learner-v1',
        'bible_pairs': len(pairs),
        'parallel_verses': len(parallel_corpus),
        'zolai_editions': ['tdb77', 'tedim1932'],
        'created': datetime.now().isoformat(),
    },
    'vocabulary': {
        'all_words': dict(all_zolai_words.most_common(500)),
        'particles': {p['word']: p for p in particle_details},
        'content_words': {w['word']: w for w in content_word_details},
        'proper_nouns': dict(categories['proper_nouns'][:100]),
    },
    'tense_markers': {
        tense: dict(words[:30])
        for tense, words in tense_markers.items()
    },
    'grammar_rules': {
        'sentence_enders': dict(grammar_rules['sentence_enders']),
        'common_prefixes': dict(grammar_rules['common_prefixes']),
        'common_suffixes': dict(grammar_rules['common_suffixes']),
        'conjunction_patterns': dict(grammar_rules['conjunction_patterns']),
        'negation_patterns': dict(grammar_rules['negation_patterns']),
        'question_patterns': dict(grammar_rules['question_patterns']),
    },
    'phrase_mapping': {
        zp: {
            'count': data['count'],
            'english': data['english'][:5],
            'examples': data['examples'][:3],
        }
        for zp, data in phrase_mapping.items()
    },
    'word_alignment': {
        word: dict(counter.most_common(5))
        for word, counter in list(word_pairs.items())[:300]
    },
    'sentence_stats': {
        'zolai_mean_length': sum(zolai_lengths) / len(zolai_lengths),
        'english_mean_length': sum(english_lengths) / len(english_lengths),
        'zolai_first_words': dict(zolai_first_words.most_common(20)),
        'zolai_last_words': dict(zolai_last_words.most_common(20)),
    },
    'crawler_config': {
        'min_score_threshold': 0.15,
        'particle_weight': 0.4,
        'vocab_weight': 0.6,
        'top_particles': [p['word'] for p in particle_details[:20]],
        'top_content_words': [w['word'] for w in content_word_details[:100]],
    },
    'training_config': {
        'text_field': 'text',
        'language': 'zolai',
        'dialect': 'tedim',
        'recommended_batch_size': 8,
        'recommended_max_length': 512,
    },
}

# Add normalization map if available
if 'normalization_map' in locals():
    language_model['normalization'] = normalization_map

# Save language model
model_file = OUTPUT_DIR / "zolai_language_model.json"
with open(model_file, 'w', encoding='utf-8') as f:
    json.dump(language_model, f, ensure_ascii=False, indent=2)

print(f"Saved language model: {model_file}")
print(f"  Vocabulary size: {len(language_model['vocabulary']['all_words'])}")
print(f"  Particles: {len(language_model['vocabulary']['particles'])}")
print(f"  Content words: {len(language_model['vocabulary']['content_words'])}")
print(f"  Tense markers: {len(language_model['tense_markers'])}")
print(f"  Grammar rules: {len(language_model['grammar_rules'])}")
print(f"  Phrase mappings: {len(language_model['phrase_mapping'])}")
print(f"  Word alignments: {len(language_model['word_alignment'])}")
print(f"  Crawler config: {len(language_model['crawler_config']['top_particles'])} particles")
print(f"  Training config: max_length={language_model['training_config']['recommended_max_length']}")

In [ ]:
#%% Cell 10: Enhanced Zolai Standardizer

# Create an enhanced standardizer using learned patterns
# This can replace or augment the existing standardizer

def create_enhanced_standardizer(language_model):
    """
    Create an enhanced text standardizer using learned language patterns.
    """
    vocab = set(language_model['vocabulary']['all_words'].keys())
    particles = set(language_model['vocabulary']['particles'].keys())
    normalization = language_model.get('normalization', {})
    
    def standardize(text):
        if not text:
            return None
        
        # Unicode normalization
        text = unicodedata.normalize('NFKC', text)
        
        # Clean whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Apply normalization map (variant -> canonical)
        words = text.split()
        normalized_words = []
        for word in words:
            w_lower = word.lower()
            if w_lower in normalization:
                normalized_words.append(normalization[w_lower])
            else:
                normalized_words.append(word)
        
        return ' '.join(normalized_words)
    
    def score_zolai(text):
        """Score how likely text is Zolai."""
        if not text or len(text) < 10:
            return 0.0
        
        words = set(text.lower().split())
        if not words:
            return 0.0
        
        # Vocabulary match
        vocab_matches = words & vocab
        particle_matches = words & particles
        
        vocab_score = len(vocab_matches) / len(words)
        particle_score = len(particle_matches) / max(len(words), 1)
        
        # Combined score
        return 0.6 * vocab_score + 0.4 * particle_score
    
    return standardize, score_zolai

# Create and test the enhanced standardizer
standardize_zolai, score_zolai = create_enhanced_standardizer(language_model)

print("Enhanced Standardizer Test:")
test_texts = [
    "A kipat cilin Pasian in vantung leh leitung a piangsak hi.",
    "This is random English text.",
    "Pasian in khuavak hoih hi ci-in mu hi.",
]

for text in test_texts:
    std = standardize_zolai(text)
    score = score_zolai(text)
    print(f"  Score: {score:.3f} | {std[:60]}...")

In [ ]:
#%% Cell 11: Export Training-Ready Dataset

# Create enhanced training data with linguistic annotations
training_data = []

for p in pairs:
    zolai_text = standardize_zolai(p['source'])
    if not zolai_text:
        continue
    
    z_words = zolai_text.lower().split()
    
    # Find tense indicators
    tense_hints = []
    for tense, markers in tense_markers.items():
        marker_words = set(w for w, _ in markers[:15])
        if marker_words & set(z_words):
            tense_hints.append(tense)
    
    # Find particles
    found_particles = [w for w in z_words if w in dict(language_model['vocabulary']['particles'])]
    
    # Find content words
    found_content = [w for w in z_words if w in dict(language_model['vocabulary']['content_words'])]
    
    # Find conjunctions
    found_conjunctions = [w for w in z_words if w in dict(grammar_rules['conjunction_patterns'])]
    
    # Calculate complexity score
    complexity = len(z_words) * 0.3 + len(found_particles) * 0.2 + len(found_content) * 0.5
    
    entry = {
        'text': zolai_text,
        'english': p['target'],
        'verse_id': p['verse_id'],
        'source_lang': p.get('src_lang', 'zolai'),
        'target_lang': p.get('tgt_lang', 'english'),
        'word_count': len(z_words),
        'particles': found_particles,
        'content_words': found_content,
        'conjunctions': found_conjunctions,
        'tense_hints': tense_hints,
        'complexity_score': round(complexity, 2),
        'language_score': score_zolai(zolai_text),
    }
    
    training_data.append(entry)

# Save annotated training data
annotated_file = OUTPUT_DIR / "zolai_annotated_training.jsonl"
with open(annotated_file, 'w', encoding='utf-8') as f:
    for entry in training_data:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Saved annotated training data: {annotated_file}")
print(f"  Entries: {len(training_data):,}")

# Stats
particle_counts = Counter()
tense_counts = Counter()
for entry in training_data:
    for p in entry['particles']:
        particle_counts[p] += 1
    for t in entry['tense_hints']:
        tense_counts[t] += 1

print(f"\nAnnotation stats:")
print(f"  Most common particles: {dict(particle_counts.most_common(15))}")
print(f"  Tense hints: {dict(tense_counts)}")
print(f"  Avg complexity: {sum(e['complexity_score'] for e in training_data)/len(training_data):.2f}")

# Also save a simple text-only version for direct training
simple_file = OUTPUT_DIR / "zolai_training_text.jsonl"
with open(simple_file, 'w', encoding='utf-8') as f:
    for entry in training_data:
        f.write(json.dumps({
            'text': entry['text'],
            'language': 'zolai',
            'source': 'bible',
        }, ensure_ascii=False) + '\n')
print(f"\nSaved simple training text: {simple_file}")

In [ ]:
#%% Cell 12: ZIP & Upload
import zipfile
from datetime import datetime

ZIP_NAME = "zolai_language_model.zip"
ZIP_PATH = WORK_DIR / ZIP_NAME

print(f"Creating ZIP: {ZIP_PATH}")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(OUTPUT_DIR.rglob("*")):
        if fpath.is_file():
            arcname = str(fpath.relative_to(OUTPUT_DIR))
            zf.write(fpath, arcname)

zip_size = ZIP_PATH.stat().st_size
print(f"\nZIP created: {ZIP_NAME} ({zip_size/1024/1024:.1f} MB)")

if IS_KAGGLE:
    print("\n" + "=" * 60)
    print("KAGGLE DATASET UPLOAD INSTRUCTIONS")
    print("=" * 60)
    print("""
Option A — Download ZIP + Upload as New Dataset:
  1. Download zolai_language_model.zip from Output panel
  2. Go to kaggle.com → Datasets → New Dataset
  3. Upload the ZIP, name it 'zolai-language-model'
  4. Set License, make it Public/Private

Option B — Use Kaggle API:
  1. Add your kaggle.json as a Kaggle Secret
  2. Set UPLOAD_TO_KAGGLE = True below

Option C — Save as Notebook Output Dataset:
  1. Save Version of this notebook
  2. The /kaggle/working/ output becomes a dataset
""")

    UPLOAD_TO_KAGGLE = False
    if UPLOAD_TO_KAGGLE:
        !pip install -q kaggle 2>/dev/null
        import kaggle
        dataset_slug = "peterpausianlian/zolai-language-model"
        print(f"Uploading to {dataset_slug}...")
        kaggle.api.dataset_create_version(
            dataset_slug,
            folder=str(OUTPUT_DIR),
            version_notes="Auto-generated by zolai-language-learner-v1"
        )
        print("Upload complete!")

# Next Steps & Reuse Guide

## What Was Learned

| Component | Description |
|-----------|-------------|
| **Tense Markers** | Zolai words that co-occur with English past/present/future markers |
| **Word Order** | Sentence structure patterns, common starters/endings |
| **Phrase Mappings** | Zolai phrases ↔ English equivalents |
| **POS Categories** | Particles, content words, proper nouns |
| **Grammar Rules** | Sentence enders, prefixes, suffixes, conjunctions, negation |
| **Word Variations** | Spelling differences between TDB77 and Tedim 1932 |

## Reuse in Other Notebooks

```python
# Load language model
with open('/kaggle/input/zolai-language-model/zolai_language_model.json') as f:
    lm = json.load(f)

# Use enhanced standardizer
from zolai_language_model import standardize_zolai, score_zolai
text = standardize_zolai(raw_text)
score = score_zolai(text)

# Use annotated training data
with open('/kaggle/input/zolai-language-model/zolai_annotated_training.jsonl') as f:
    training = [json.loads(line) for line in f]
```

## Next Steps

1. **Update Crawler** — Use `score_zolai()` from language model for better filtering
2. **Fine-Tune Qwen** — Use `zolai_annotated_training.jsonl` with linguistic features
3. **Expand Vocabulary** — Add more sources (hymns, literature, web content)
4. **Build Dictionary** — Use phrase mappings for Zolai-English dictionary
5. **Grammar Checker** — Use grammar rules to validate Zolai text